# TSV RLCK Iteration 2 Demo

This notebook demonstrates single-point extraction, frequency sweep, Monte Carlo analysis, and benchmark error reporting.

In [ ]:
import json
from pathlib import Path
import sys

root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(root / 'src'))

from tsv_rlck.models import TSVGeometry, TSVProcess, OperatingPoint, SweepConfig, MonteCarloConfig
from tsv_rlck.extractor import extract_single_tsv, extract_sweep_tsv, run_monte_carlo_tsv
from tsv_rlck.benchmark import compare_single_against_reference, summarize_error, compare_batch_against_reference
from tsv_rlck.sparams import extract_sparams, to_touchstone_s2p

In [ ]:
geometry = TSVGeometry(diameter_um=10.0, height_um=60.0, oxide_thickness_nm=200.0, pitch_um=40.0)
process = TSVProcess()
op = OperatingPoint(frequency_hz=1e9, temperature_c=25.0, bias_v=0.0)

single = extract_single_tsv(geometry, process, op)
single

In [ ]:
sweep_cfg = SweepConfig(f_start_hz=1e6, f_stop_hz=1e10, points=21, spacing='log')
sweep = extract_sweep_tsv(geometry, process, op, sweep_cfg)
{k: (v[:3], v[-3:]) if isinstance(v, list) else v for k, v in sweep.items()}

In [ ]:
mc_cfg = MonteCarloConfig(
    mc_samples=100,
    seed=42,
    sigma_diameter_um=0.1,
    sigma_oxide_thickness_nm=2.0,
    sigma_metal_resistivity_ohm_m=1e-10,
    sigma_substrate_resistivity_ohm_cm=0.2,
    sigma_temperature_c=2.0,
)
mc = run_monte_carlo_tsv(geometry, process, op, mc_cfg)
mc['statistics']

In [ ]:
benchmark_path = root / 'benchmarks' / 'example_reference.json'
benchmark_data = json.loads(benchmark_path.read_text(encoding='utf-8'))
reference = benchmark_data['reference']

report = compare_single_against_reference(geometry, process, op, reference)
summary = summarize_error(report)
report, summary

In [ ]:
batch_path = root / 'benchmarks' / 'batch_reference.json'
batch_data = json.loads(batch_path.read_text(encoding='utf-8'))
batch_report = compare_batch_against_reference(batch_data['cases'])
batch_report['overall']

In [ ]:
op_adv = OperatingPoint(frequency_hz=1e9, temperature_c=25.0, bias_v=0.0, model_level='advanced')
sparam_sweep = SweepConfig(f_start_hz=1e6, f_stop_hz=1e10, points=21, spacing='log')
sparams = extract_sparams(geometry, process, op_adv, sweep=sparam_sweep, z0=50.0)

s2p_path = root / 'benchmarks' / 'demo_output.s2p'
s2p_path.write_text(to_touchstone_s2p(sparams), encoding='utf-8')

{
    'points': len(sparams['points']),
    'first_point': sparams['points'][0],
    's2p_path': str(s2p_path)
}

## S-parameter extraction and Touchstone export

Compute two-port S-parameters from RLCK over sweep frequencies and write an `.s2p` file.

## Batch benchmark report

Load `benchmarks/batch_reference.json` and generate overall metrics for multiple cases.